# 使用@tool装饰器定义工具

## 1、自定义工具描述：description

举例1：函数使用@tool装饰器修饰以后，就是一个可以被模型识别的工具

如下的程序报错了，因为：在没有提供description参数的情况下，要求函数必须提供docstring

ValueError: Function must have a docstring if description not provided.

In [3]:
from langchain_core.tools import tool
from langchain_core.utils.function_calling import convert_to_openai_tool
from rich import print as rprint


@tool
def get_weather(city: str):
    return f"{city}天气晴朗~~"

rprint(convert_to_openai_tool(get_weather))

ValueError: Function must have a docstring if description not provided.

修改为：

In [4]:
from langchain_core.tools import tool
from langchain_core.utils.function_calling import convert_to_openai_tool
from rich import print as rprint

@tool
def get_weather(city: str):
    """获取城市的天气"""
    return f"{city}天气晴朗~~"

rprint(convert_to_openai_tool(get_weather))

{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': '获取城市的天气',
        'parameters': {'properties': {'city': {'type': 'string'}}, 'required': ['city'], 'type': 'object'}
    }
}

举例2：使用description参数

In [5]:
from langchain_core.tools import tool
from langchain_core.utils.function_calling import convert_to_openai_tool
from rich import print as rprint

@tool(description="获取具体城市的天气情况")
def get_weather(city: str):
    return f"{city}天气晴朗~~"

rprint(convert_to_openai_tool(get_weather))

{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': '获取具体城市的天气情况',
        'parameters': {'properties': {'city': {'type': 'string'}}, 'required': ['city'], 'type': 'object'}
    }
}

举例3：在同时声明了description和docstring的情况下，description的优先级更高

In [6]:
from langchain_core.tools import tool
from langchain_core.utils.function_calling import convert_to_openai_tool
from rich import print as rprint

@tool(description="获取具体城市的天气情况")
def get_weather(city: str):
    """获取城市的天气"""
    return f"{city}天气晴朗~~"

rprint(convert_to_openai_tool(get_weather))

{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': '获取具体城市的天气情况',
        'parameters': {'properties': {'city': {'type': 'string'}}, 'required': ['city'], 'type': 'object'}
    }
}

举例4：解析`docstring：parse_docstring`

当我们没有向 `@tool` 传递 `description` 参数时，默认情况下， `tool` 会将 `docstring` 整体视为
`description`

要注意：不使用 `@tool` 装饰器时，`docstring`不合法会被视为普通文本，作为 `description` ，但如果使
用了 `@tool` 时 `docstring` 不合法，将会抛出异常

In [12]:
from langchain_core.tools import tool
from langchain_core.utils.function_calling import convert_to_openai_tool
from rich import print as rprint

@tool(parse_docstring=True)
def get_weather(city: str):
    """
    获取城市的天气

    Args:
        city : 城市
    """
    return f"{city}天气晴朗~~"

rprint(convert_to_openai_tool(get_weather))

{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': '获取城市的天气',
        'parameters': {
            'properties': {'city': {'description': '城市', 'type': 'string'}},
            'required': ['city'],
            'type': 'object'
        }
    }
}

举例5：

In [13]:
from langchain_core.tools import tool
from langchain_core.utils.function_calling import convert_to_openai_tool
from rich import print as rprint

@tool(parse_docstring=True, description="获取具体城市的天气")
def get_weather(city: str):
    """
    获取城市的天气

    Args:
        city : 城市
    """
    return f"{city}天气晴朗~~"

rprint(convert_to_openai_tool(get_weather))

{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': '获取具体城市的天气',
        'parameters': {
            'properties': {'city': {'description': '城市', 'type': 'string'}},
            'required': ['city'],
            'type': 'object'
        }
    }
}

## 2、更改工具名称：name_or_callable

说明：不要使用config或runtime作为参数名，这些是LangChain内部保留的。

开发中，习惯使用函数名作为工具名称，**不推荐**自定义工具名称。

举例：

In [14]:
from langchain_core.tools import tool
from langchain_core.utils.function_calling import convert_to_openai_tool
from rich import print as rprint

@tool(parse_docstring=True, name_or_callable="GetWeather")
def get_weather(city: str):
    """
    获取城市的天气

    Args:
        city : 城市
    """
    return f"{city}天气晴朗~~"

rprint(convert_to_openai_tool(get_weather))

{
    'type': 'function',
    'function': {
        'name': 'GetWeather',
        'description': '获取城市的天气',
        'parameters': {
            'properties': {'city': {'description': '城市', 'type': 'string'}},
            'required': ['city'],
            'type': 'object'
        }
    }
}

## 3、自定义args_schema

### 3.1 使用Pydantic模型定义

当工具的参数变得复杂，需要 枚举值 、 范围限制 或 更复杂的业务逻辑验证 时，Pydantic 模型是理想
的选择，提供强大的类型检查和数据验证。


举例1：pydantic类型的定义

In [17]:
from pydantic import BaseModel
from langchain_core.tools import tool
from langchain_core.utils.function_calling import convert_to_openai_tool
from rich import print as rprint


class WeatherInput(BaseModel):
    city : str

@tool(args_schema=WeatherInput)
def get_weather(city: str):
    """
    获取城市的天气
    """
    return f"{city}天气晴朗~~"

rprint(convert_to_openai_tool(get_weather))

{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': '获取城市的天气',
        'parameters': {'properties': {'city': {'type': 'string'}}, 'required': ['city'], 'type': 'object'}
    }
}

举例2：Field() ：用来“ 定制字段 ”的函数，可用于设置默认值、描述等。


In [22]:
from pydantic import BaseModel, Field
from langchain_core.tools import tool
from langchain_core.utils.function_calling import convert_to_openai_tool
from rich import print as rprint


class WeatherInput(BaseModel):
    city : str = Field(
        description="具体的城市",
        default="仲恺",
    )

@tool(args_schema=WeatherInput)
def get_weather(city: str):
    """
    获取城市的天气
    """
    return f"{city}天气晴朗~~"

rprint(convert_to_openai_tool(get_weather))

{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': '获取城市的天气',
        'parameters': {
            'properties': {'city': {'default': '仲恺', 'description': '具体的城市', 'type': 'string'}},
            'type': 'object'
        }
    }
}

举例3：可以使用 Literal类型限定参数为固定选项。

`Literal` ：表示字段不能是任意某种类型的值，而只能是几个固定字面量之一。

In [29]:
from typing import Literal

class WeatherInput(BaseModel):
    city : str = Field(
        description = "具体的城市",
        default = "仲恺",
    )
    unit : Literal["Celsius", "Fahrenheit"] = Field(
        default = "Celsius",
        description = "气温单位"
    ) # 枚举
    include_forecast : bool = Field(
        default = False,
        description = "是否包含未来五天的天气预报"
    ) # 布尔类型

@tool(args_schema=WeatherInput)
def get_weather(city: str, unit: str = "Fahrenheit", include_forecast: bool = True) -> str:
    """
    获取城市的天气
    """
    return f"{city}天气晴朗~~"
rprint(convert_to_openai_tool(get_weather))

{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': '获取城市的天气',
        'parameters': {
            'properties': {
                'city': {'default': '仲恺', 'description': '具体的城市', 'type': 'string'},
                'unit': {
                    'default': 'Celsius',
                    'description': '气温单位',
                    'enum': ['Celsius', 'Fahrenheit'],
                    'type': 'string'
                },
                'include_forecast': {
                    'default': False,
                    'description': '是否包含未来五天的天气预报',
                    'type': 'boolean'
                }
            },
            'type': 'object'
        }
    }
}

### 3.2 使用Json Schema定义

在 LangChain 中，还可以直接使用 JSON Schema 字典 来定义工具的参数模式。这种方式提供了极大的灵活性。

举例：

In [30]:

json_schema = {
    'properties': {
        'city': {'default': '仲恺', 'description': '输入的具体的城市', 'type': 'string'},
        'unit': {
            'default': 'Celsius',
            'description': '气温单位',
            'enum': ['Celsius', 'Fahrenheit'],
            'type': 'string'
        },
        'include_forecast': {
            'default': False,
            'description': '是否包含未来五天的天气预报',
            'type': 'boolean'
        }
    },
    'type': 'object'
}


@tool(args_schema=json_schema)
def get_weather(city: str, unit: str = "Fahrenheit", include_forecast: bool = True) -> str:
    """
    获取城市的天气
    """
    return f"{city}天气晴朗~~"
rprint(convert_to_openai_tool(get_weather))

{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': '获取城市的天气',
        'parameters': {
            'properties': {
                'city': {'default': '仲恺', 'description': '输入的具体的城市', 'type': 'string'},
                'unit': {
                    'default': 'Celsius',
                    'description': '气温单位',
                    'enum': ['Celsius', 'Fahrenheit'],
                    'type': 'string'
                },
                'include_forecast': {
                    'default': False,
                    'description': '是否包含未来五天的天气预报',
                    'type': 'boolean'
                }
            },
            'type': 'object'
        }
    }
}